# Currently meant to run locally - adjust path as needed!!!!!

In [23]:
import os
#test locally - PoC
# where it's running
current_dir = os.getcwd()
print(f"you are in: {current_dir}")

# folder:
audio_folder = "../../raw_data/audio_and_txt_files/" 

if os.path.exists(audio_folder):
    files = [f for f in os.listdir(audio_folder) if f.endswith('.wav')]
    print(f"{len(files)} files .wav")
else:
    print("not found")

you are in: /home/totid/code/antonella-dm/projeto/notebooks/personal_work
920 files .wav


In [24]:
import librosa
import numpy as np
import pandas as pd
from tqdm import tqdm #shows progress
import os
import sys

# FEATURES

In [25]:
def extract_features_raw(y, sr):
    feat = {}
     # TEMPORAL
    feat['rms_mean'] = float(np.mean(librosa.feature.rms(y=y)))
    feat['zcr_mean'] = float(np.mean(librosa.feature.zero_crossing_rate(y=y)))
    
    # SPECTRAL
    feat['centroid_mean'] = float(np.mean(librosa.feature.spectral_centroid(y=y, sr=sr)))
    feat['rolloff_mean'] = float(np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr)))
    feat['flatness_mean'] = float(np.mean(librosa.feature.spectral_flatness(y=y)))
    feat['flux_mean'] = float(np.mean(librosa.onset.onset_strength(y=y, sr=sr)))
    
    # MFCC (12, ignore 0) 
    mfccs = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13) # 12 mfccs + other 6 = 18 total
    for i in range(1, 13):
        feat[f'mfcc_{i}'] = float(np.mean(mfccs[i]))
        
    return feat

# SANITY CHECK

In [27]:
# sanit xheck
print("sanit check")

# white noise
mock_sr = 10000 #instead of the std 22050Hz
mock_y = np.random.uniform(-1, 1, mock_sr * 6)

#
test_feat = extract_features_raw(mock_y, mock_sr)

# chekc
errors = []
if not isinstance(test_feat, dict): errors.append("must be a dict")
if len(test_feat) != 18: errors.append(f"Esperado 18 colunas, mas recebi {len(test_feat)}.")
if any(np.isnan(list(test_feat.values()))): errors.append("NaN in features")

if not errors:
    print("done - 18 vlid columns")
    # example
    print(f"MFCC_1 example: {test_feat.get('mfcc_1'):.4f}")
    print(f"RMS example: {test_feat.get('rms_mean'):.4f}")
else:
    print("errors:")
    for err in errors: print(f" - {err}")

sanit check
done - 18 vlid columns
MFCC_1 example: -2.3594
RMS example: 0.5720


# MAP

In [28]:
# raw_data audios
diagnosis_path = "../../raw_data/patient_diagnosis.csv" 

if os.path.exists(diagnosis_path):
    # adjust names CSV
    df_diag = pd.read_csv(diagnosis_path, names=['patient_id', 'diagnosis'])
    
    # create a dict for ultra-fast lookup: { '101': 'COPD', ... }
    diag_map = dict(zip(df_diag['patient_id'].astype(str), df_diag['diagnosis']))
    print(f"mapping created for {len(diag_map)} patients.")
else:
    print(f"not found")
    diag_map = {}

mapping created for 126 patients.


In [29]:
# configuration
TARGET_SR = 10000 # instead of the std 20050 
CHUNK_DURATION = 6 
all_results = []

# using all files
print(f"processing {len(files)} files...")

for f in tqdm(files):
    path = os.path.join(audio_folder, f)
    
    try:
        # PREPROCESSING
        
        # Load + Resample + Normalization? check
        # standardizes all audio to 22050Hz and scales amplitude to [-1, 1]
        y, sr = librosa.load(path, sr=TARGET_SR)
        
        # Cleaning - trim
        # removes silent intervals from the beginning and end
        y, _ = librosa.effects.trim(y)
        
        # slicing (6s windows)
        # breaks the cleaned audio into fixed 6-second segments
        samples_per_chunk = CHUNK_DURATION * TARGET_SR
        chunks = [y[i : i + samples_per_chunk] for i in range(0, len(y), samples_per_chunk) 
                  if len(y[i : i + samples_per_chunk]) == samples_per_chunk]
        
        # FEATURE EXTRACTION - this cell is getting too long - maybe adjust it later
        
        # extract ID from the filename (e.g., '101')
        p_id = f.split('_')[0]
        
        for i, chunk in enumerate(chunks):
            # extract 18 features from the 6s processed chunk
            features = extract_features_raw(chunk, TARGET_SR) # it calculates the mean for the 6s slice and return the 18 features
            
            # add metadata and labels
            features['patient_id'] = p_id
            features['chunk_id'] = i
            features['original_file'] = f
            
            # map diagnosis from our diag_map (fallback to 'unknown')
            features['diagnosis'] = diag_map.get(p_id, "Unknown")
            
            all_results.append(features)
            
    except Exception as e:
        print(f"error on{f}: {e}")

# final df
df_final = pd.DataFrame(all_results)

# check
print(f"\ndone - dataset w {len(df_final)} rows.")
print(f"unique diagnoses found: {df_final['diagnosis'].unique()}")

processing 920 files...


100%|████████████████████████████████████████████████| 920/920 [01:12<00:00, 12.62it/s]


done - dataset w 2978 rows.
unique diagnoses found: ['COPD' 'URTI' 'Pneumonia' 'Healthy' 'Bronchiolitis' 'Bronchiectasis'
 'LRTI' 'Asthma']


In [30]:
display(df_final.head(15))

,rms_mean,zcr_mean,centroid_mean,rolloff_mean,flatness_mean,flux_mean,mfcc_1,mfcc_2,mfcc_3,mfcc_4,...,mfcc_7,mfcc_8,mfcc_9,mfcc_10,mfcc_11,mfcc_12,patient_id,chunk_id,original_file,diagnosis
0,0.362614,0.001539,48.431325,44.110832,0.000012,0.543730,73.807640,50.533783,39.836922,37.324631,...,23.810909,20.117891,14.189551,9.281930,8.831337,11.688478,223,0,223_1b1_Pr_sc_Meditron.wav,COPD
1,0.302596,0.001879,57.962129,50.690215,0.000098,0.615449,83.702332,44.393806,32.717323,37.988594,...,21.217758,23.032206,16.925091,8.563718,8.135166,12.413423,223,1,223_1b1_Pr_sc_Meditron.wav,COPD
2,0.262564,0.001899,53.089966,42.372881,0.000015,0.472277,72.223984,48.411705,40.204586,38.110851,...,23.663528,21.205318,14.382548,7.264541,6.590406,10.875096,223,2,223_1b1_Pr_sc_Meditron.wav,COPD
3,0.259362,0.001382,47.320406,33.103814,0.000002,0.502166,78.755859,51.148075,41.824574,41.196419,...,25.118649,21.782694,13.541658,5.844097,5.610048,10.750846,223,3,223_1b1_Pr_sc_Meditron.wav,COPD
4,0.199427,0.001432,53.594167,40.138374,0.000006,0.540275,95.019043,50.130978,36.215141,43.793114,...,22.806818,23.043684,15.013694,3.548583,1.795268,10.379703,223,4,223_1b1_Pr_sc_Meditron.wav,COPD
5,0.133809,0.002859,72.873357,60.083422,0.000020,0.442445,92.079819,70.158600,52.474094,39.188721,...,17.904701,13.914968,11.966480,10.659824,9.976493,9.574545,134,0,134_2b2_Al_mc_LittC2SE.wav,COPD
6,0.133879,0.003228,72.856174,58.842029,0.000025,0.393108,85.389717,68.509041,52.921295,40.572254,...,18.824087,14.945706,12.431826,10.743546,9.681673,9.179729,134,1,134_2b2_Al_mc_LittC2SE.wav,COPD
7,0.140084,0.003083,67.433068,48.083289,0.000008,0.410735,89.673897,70.452072,52.545193,39.267502,...,18.091009,14.598758,12.291980,10.800585,10.050264,9.675508,134,2,134_2b2_Al_mc_LittC2SE.wav,COPD
8,0.155688,0.038069,699.499975,1652.542373,0.006866,1.177726,82.183342,16.375933,34.929325,14.027961,...,18.472412,4.706385,10.842641,5.565956,10.499599,2.936414,170,0,170_1b4_Al_mc_AKGC417L.wav,COPD
9,0.150935,0.027431,576.645072,1321.586997,0.002722,0.887159,95.111618,12.059289,28.866213,16.906809,...,19.917177,6.897327,10.482384,6.255206,9.397890,4.087189,170,1,170_1b4_Al_mc_AKGC417L.wav,COPD


In [31]:
print(df_final.columns.tolist())

['rms_mean', 'zcr_mean', 'centroid_mean', 'rolloff_mean', 'flatness_mean', 'flux_mean', 'mfcc_1', 'mfcc_2', 'mfcc_3', 'mfcc_4', 'mfcc_5', 'mfcc_6', 'mfcc_7', 'mfcc_8', 'mfcc_9', 'mfcc_10', 'mfcc_11', 'mfcc_12', 'patient_id', 'chunk_id', 'original_file', 'diagnosis']


In [32]:
print(df_final['diagnosis'].value_counts().head())

diagnosis
COPD              2597
Pneumonia          111
Healthy            105
URTI                69
Bronchiectasis      48
Name: count, dtype: int64


In [22]:
# Save CSV to raw_data
df_final.to_csv("../../raw_data/xgboost_features_6s_final.csv", index=False)
print("done")

done
